# About this notebook

This notebook takes the statement of work (SoW) topics generated by the LLM and merged with the MA dataset.  The filtered_topics csv and llm_ma_merge csv files are generated by the SoW_labels notebook.  It then clusters the topics and assigns a label that can be used for further analysis.  Review of the topics was done manually by a human familiar with FEMA mission assignments.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from top2vec import Top2Vec
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

# Use filtered topics from LLM to create a dataframe with documents and SoW topics using top2vec

In [11]:
df=pd.read_csv('filtered_topics.csv')

In [12]:
df.head()

,Unnamed: 0,maId,clean_statement_topic,declarationType,maAmendNumber,supportFunction
0,0,3612EMCTDOI-USGS01,flood monitoring and data collection,EM,0,5.0
1,10,4856DRCAUSDA-APH01,personnel support,DR,0,11.0
2,16,4795DRNMDOT01,personnel support,DR,0,1.0
3,18,3614EMLADOT01,personnel support,EM,0,1.0
4,31,3614EMLADOJ-ATF01,personnel support,EM,0,13.0


In [38]:
MA_df=df[(df['declarationType']!='SU')&(df['maAmendNumber']==0)&(df['supportFunction']<=15)]

In [39]:
sow_list = df['clean_statement_topic'].to_list()

In [15]:
model=Top2Vec(sow_list)
#model=Top2Vec.load('clean_topics_model')

In [20]:
model.save('clean_topics_model')

In [40]:
topic_sizes, topic_nums = model.get_topic_sizes()

In [41]:
topic_list=topic_nums.tolist()
size_list=topic_sizes.tolist()

In [42]:
len(topic_list)

94

In [43]:
#create a dictionary with topic numbers as keys, and documents as values.
topic_dict={}
for t in topic_list:
    documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=t, num_docs=size_list[t])
    doc_list=document_ids.tolist()
    for doc in doc_list:
        topic_dict[doc]=t

In [44]:
key_list=list(topic_dict.keys())
key_list.sort()

sort_topic = {i: topic_dict[i] for i in key_list}

In [45]:
value_list = list(sort_topic.values())

In [46]:
df['clean_sow_topic']=value_list

In [57]:
df.head(10)

,Unnamed: 0,maId,clean_statement_topic,declarationType,maAmendNumber,supportFunction,clean_sow_topic
0,0,3612EMCTDOI-USGS01,flood monitoring and data collection,EM,0,5.0,78
1,10,4856DRCAUSDA-APH01,personnel support,DR,0,11.0,0
2,16,4795DRNMDOT01,personnel support,DR,0,1.0,0
3,18,3614EMLADOT01,personnel support,EM,0,1.0,0
4,31,3614EMLADOJ-ATF01,personnel support,EM,0,13.0,0
5,35,4856DRCADOJ-ATF01,personnel support,DR,0,13.0,0
6,49,3614EMLADOE-OE02,energy support operations,EM,0,12.0,86
7,51,3621EMVADOE-OE01,energy support operations,EM,0,12.0,86
8,56,4856DRCADOE-OE01,energy support operations,DR,0,12.0,86
9,57,3624EMKYDOE-OE01,energy support operations,EM,0,12.0,86


In [26]:
df.to_csv('clean_topics_top2vec.csv')

# Create combined csv file with both SoW and assistance requested (AR) topics

This requires first running the MA_topics notebook to obtain the AR_topics csv.

In [27]:
AR_df=pd.read_csv('AR_topics.csv')

In [28]:
merged=pd.merge(df,AR_df,how='left',on='maId')

In [29]:
merged.to_csv('top2vecsow_AR.csv')

# Join LLM topics to top2vec clusters to get SOW 

The following series of cells combines the LLM filtered topics with the top2vec cluster numbers.  It then selects the topic that is 'most representative' of the cluster using top2vec's document search built-in method.  For several clusters which returned empty documents, the document was assigned manually in the resulting csv by a human familiar with FEMA's mission assignment process.

In [75]:
llm=pd.read_csv('llm_MA_merge.csv')

In [76]:
llm_filter=llm[(llm['declarationType']!='SU')&(llm['maAmendNumber']==0)&(llm['supportFunction']<=15)]

In [77]:
llm_sow=llm[['maId','clean_statement_actions_extracted','clean_statement_topic']]

In [78]:
llm_sow.drop_duplicates('maId',inplace=True)

C:\Users\sbpow\AppData\Local\Temp\ipykernel_16728\368446321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  llm_sow.drop_duplicates('maId',inplace=True)


In [79]:
len(llm_sow)

10996

In [80]:
df_filter=df[['maId','clean_sow_topic']]

In [95]:
new_df=llm_sow.merge(df_filter,how='right',on='maId')

In [96]:
new_df.head()

,maId,clean_statement_actions_extracted,clean_statement_topic,clean_sow_topic
0,3612EMCTDOI-USGS01,"[""Provide advance support for flood event oper...",flood monitoring and data collection,78
1,4856DRCAUSDA-APH01,"[""Provide personnel to the RRCC, IOF, JFO, or ...",personnel support,0
2,4795DRNMDOT01,"[""Provide appropriate personnel to RRCC, IOF, ...",personnel support,0
3,3614EMLADOT01,"[""Provide appropriate personnel to RRCC, IOF, ...",personnel support,0
4,3614EMLADOJ-ATF01,"[""Provide appropriate personnel to RRCC, JFO, ...",personnel support,0


In [97]:
#find document that is closest to the actual topic and append to dictionary
topic_dict={}
for t in topic_list:
    documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=t, num_docs=1)
    doc_list=documents.tolist()
    doc_ids=document_ids.tolist()
    for doc in doc_ids:
        topic_dict[t]=new_df['clean_statement_actions_extracted'].iloc[doc]

In [92]:
new_df[new_df['clean_sow_topic']==86]

,maId,clean_statement_actions_extracted,clean_statement_topic,clean_sow_topic
6,3614EMLADOE-OE02,"[""Provide appropriate personnel to the RRCC, I...",energy support operations,86
7,3621EMVADOE-OE01,"[""Provide appropriate personnel to the RRCC, I...",energy support operations,86
8,4856DRCADOE-OE01,"[""Provide appropriate personnel to the RRCC, I...",energy support operations,86
9,3624EMKYDOE-OE01,"[""Provide appropriate personnel to the RRCC, I...",energy support operations,86
10,4860DRKYDOE-OE01,"[""Provide appropriate personnel to the RRCC, I...",energy support operations,86
1663,3563EMRIDOE-OE01,"[""Provide appropriate personnel to the RRCC, I...",energy support operations,86
1806,4570DRLADOE02,"[""Provide personnel to the RRCC, IOF, JFO, or ...",energy support operations,86
1947,4558DRCADOE01,"[""Provide personnel to the RRCC, IOF, JFO, or ...",energy support operations,86
2043,3532EMPRDOE01,"[""Provide personnel to the RRCC, IOF, JFO, or ...",energy support operations,86
3293,3535EMCTDOE01,[],NaN,86


In [103]:
#create a dataframe with the 'most representative' documents
sow_topics=pd.DataFrame(list(topic_dict.values()),index=list(topic_dict.keys()))

In [104]:
sow_topics.head()

,0
0,"[""Provide personnel to support FEMA disaster o..."
1,"[""Deploy personnel to support recovery efforts..."
2,"[""Provide personnel to FEMA R2 RRCC."", ""Provid..."
3,"[""Provide personnel to the RRCC/Field and othe..."
4,[]


In [106]:
sow_topics.to_csv('sow_topic_cluster_centers.csv')